In [1]:
import numpy as np
import torch

In [2]:
import torchvision
from torchvision import transforms

transform = transforms.Compose([
             transforms.ToTensor(),
             transforms.Normalize((.5,), (.5,))
])

train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

trainLoader = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
testLoaderr = torch.utils.data.DataLoader(test, batch_size=128, shuffle=False)




100%|██████████| 26.4M/26.4M [00:02<00:00, 10.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 165kB/s]
100%|██████████| 4.42M/4.42M [00:03<00:00, 1.30MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 21.5MB/s]


In [3]:
train.classes

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [4]:
sample_img, sample_label = next(iter(trainLoader))
print(sample_label.shape)
sample_img.shape

torch.Size([128])


torch.Size([128, 1, 28, 28])

In [10]:
import torch.nn as nn
import torch.nn.functional as F
class MyCNN(nn.Module):
    def __init__(self, ):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64*5*5, 512)
        self.fc2 = nn.Linear(512, 10)



    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))

        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))

        x = self.fc2(x)
        return x

In [11]:
model = MyCNN()

print(model)

MyCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1600, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
)


In [12]:
dummy = torch.randn(128, 1, 28, 28)
model(dummy)

tensor([[-0.0092, -0.0079, -0.1041,  ...,  0.0119,  0.0333, -0.1306],
        [ 0.0178,  0.0801, -0.0495,  ...,  0.0060,  0.0319, -0.1205],
        [ 0.0421,  0.0156, -0.0035,  ..., -0.0006,  0.0215, -0.1499],
        ...,
        [-0.0109, -0.0092, -0.0551,  ..., -0.0155,  0.0087, -0.0835],
        [ 0.0068,  0.0210, -0.0653,  ...,  0.0214,  0.0109, -0.0847],
        [ 0.0089,  0.0629, -0.0088,  ..., -0.0247,  0.0114, -0.0907]],
       grad_fn=<AddmmBackward0>)

In [21]:
#Training loop
from torch import optim
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fun = nn.CrossEntropyLoss()

for epoch in range(2):
    correct = 0
    total = 0
    for img, tar in trainLoader:
        optimizer.zero_grad()
        output = model(img)

        loss = loss_fun(output, tar)
        loss.backward()

        optimizer.step()

        _, pred = torch.max(output, 1)

        correct += (pred == tar).sum().item()
        total += tar.shape[0]

    acc = (correct/total) *100
    print(acc)


85.35166666666667
86.56333333333333


In [22]:
#Testing loop
total_test = 0
correct_test = 0
with torch.no_grad():
 for img, label in testLoaderr:
    output_test = model(img)
    _, pred = torch.max(output_test, 1)
    correct_test += (pred == label).sum().item()
    total_test += label.shape[0]


acc = (correct_test/total_test) * 100
print(acc)



85.65


In [23]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=28, in_channel=1, patch_size=7, embed_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.num_patch = (img_size//patch_size)**2

        self.projection = nn.Linear(in_channel*self.patch_size*self.patch_size, embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape

        x = x.unfold(2, self.patch_size, self.patch_size) \
            .unfold(3, self.patch_size, self.patch_size)

        x = x.contiguous().view(B, self.num_patch , -1)
        return self.projection(x)


In [24]:
import torch
class MyVisionTransformer(nn.Module):
    def __init__(self, img_size=28, patch_size=7, in_channel=1, num_heads = 4, num_layers = 4, embed_dim=64, num_classes = 10):
        super().__init__()
        self.patch_class = PatchEmbedding(img_size, in_channel, patch_size, embed_dim)
        self.num_patches = (img_size // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.zeros(1,1,embed_dim))

        self.position_embed = nn.Parameter(torch.zeros(1, self.num_patches+1, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=128, batch_first=True)

        self.transformer = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)

        self.head = nn.Linear(embed_dim, num_classes)


    def forward(self, x):
        B = x.shape[0]

        x = self.patch_class(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        x += self.position_embed
        x = self.transformer(x)

        cls_output = x[:, 0]
        return self.head(cls_output)




In [25]:
model = MyVisionTransformer()
print(model)

MyVisionTransformer(
  (patch_class): PatchEmbedding(
    (projection): Linear(in_features=49, out_features=64, bias=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (head): Linear(in_features=64, out_features=10, bias=True)
)


In [26]:
fake_input = torch.randn(8, 1, 28, 28)
out = model(fake_input)
out


tensor([[-0.0128,  0.3596, -1.0747, -1.1276,  0.7774,  0.0095,  0.9667, -0.6896,
          0.4471,  0.8296],
        [-0.3069,  0.7869, -0.7331,  0.7034, -0.2441,  0.4349, -0.4014, -0.2518,
         -1.1028,  0.0569],
        [-1.0046,  0.1474,  0.1405, -0.4300,  0.2698, -0.3693, -0.1315, -0.2951,
          0.5104,  1.0698],
        [-1.1772,  1.2502, -0.1890,  0.3551, -0.4251,  0.5101, -0.6530, -0.4161,
         -0.6745, -0.1246],
        [-0.1771,  0.3485,  0.4679, -0.0977, -0.4274, -0.3302,  0.2617,  0.5075,
          0.1312,  0.9169],
        [ 0.0317,  0.5759, -0.0972,  0.1378,  0.0857,  0.4328, -0.5996,  0.4424,
         -0.5353, -0.0430],
        [-0.8307,  0.4483, -0.2263, -0.0794,  0.0676, -0.3562, -0.6531,  0.0634,
         -0.1938,  0.0024],
        [ 0.6038, -0.7166, -1.2638,  0.2265,  0.2035,  0.3580, -0.7016, -0.2097,
         -0.1578,  0.5185]], grad_fn=<AddmmBackward0>)

In [27]:
entropy = nn.CrossEntropyLoss()
optimizer_transformer = optim.Adam(model.parameters(), lr=0.001)

for _ in range(2):
    total = 0
    correct = 0
    for img, label in trainLoader:
        optimizer_transformer.zero_grad()
        output = model(img)
        loss = entropy(output, label)
        loss.backward()

        optimizer_transformer.step()
        _, pred = torch.max(output, 1)

        correct += (pred == label).sum().item()
        total += label.size(0)

    acc = (correct/total) * 100
    print(acc)



In [30]:
total_trans_test = 0
correct_trans_test = 0
with torch.no_grad():
    for img_test, label_test in testLoaderr:
        output_trans_test = model(img_test)

        _, pred_test = torch.max(output_trans_test, 1)

        correct_trans_test += (pred_test == label_test).sum().item()
        total_trans_test += label_test.shape[0]

acc = (correct_trans_test/total_trans_test) * 100
print(acc)

82.32000000000001
